# Bitki Hastalik Tespiti - Colab Egitim
Bu notebook, PlantVillage veri seti ile modeli egitir ve ONNX export alir.

**Beklenen yapilar:**
- Drive: `/content/drive/MyDrive/PlantVillage` (ham veri)
- Drive: `/content/drive/MyDrive/bitki-hastalik-app` (proje klasoru)


## 1) Google Drive bagla


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Projeyi calisma dizinine kopyala
Drive icindeki proje klasorunu Colab calisma alanina kopyalar.


In [ ]:
import os, shutil
SRC = '/content/drive/MyDrive/bitki-hastalik-app'
DST = '/content/bitki-hastalik-app'
if not os.path.exists(SRC):
    raise SystemExit('Drive icine bitki-hastalik-app klasorunu yukleyin.')
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print('Kopyalandi:', DST)


## 3) Gereksinimler


In [ ]:
%cd /content/bitki-hastalik-app
!pip install -r ml/requirements.txt
!pip install onnx onnxscript


## 4) Veri hazirligi (tum siniflar)


In [ ]:
!python ml/prepare_data.py \
  --raw-dir /content/drive/MyDrive/PlantVillage \
  --out-dir ml/data/plantvillage \
  --val-ratio 0.2


## 5) Egitim (GPU oneri)


In [ ]:
!python ml/train.py \
  --data-dir ml/data/plantvillage \
  --epochs 12 \
  --batch-size 32 \
  --img-size 224 \
  --model efficientnet_v2_s \
  --pretrained \
  --out-dir ml/artifacts


## 6) ONNX Export


In [ ]:
!python ml/export_onnx.py \
  --checkpoint ml/artifacts/best.pt \
  --img-size 224 \
  --out ml/artifacts/model.onnx


## 7) Cikti dosyalari
Drive icine kaydedilmesi icin kopyalar.


In [ ]:
!cp ml/artifacts/best.pt /content/drive/MyDrive/bitki-hastalik-app/server/model/model.pt
!cp ml/artifacts/model.onnx /content/drive/MyDrive/bitki-hastalik-app/server/model/model.onnx
!cp ml/artifacts/labels.json /content/drive/MyDrive/bitki-hastalik-app/server/model/labels.json
print('Kopyalandi: server/model dosyalari')
